# Multimodal fusion

**Research objective.** Compare saved weighted late fusion with the archived CLIP gated-fusion checkpoint.

All results are read from local FNEWS artifacts. `SOURCE_TRUE` and `SOURCE_NON_TRUE` are Fakeddit source-taxonomy labels, not independently verified factuality labels. The held-out test partition is not opened by these notebooks except where a prior frozen metric is explicitly quoted.

In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd()
if not (ROOT / "data/research_dataset").exists():
    ROOT = Path("/Users/aka_ank/Desktop/FNEWS")
DATA = ROOT / "data/research_dataset"
REPORTS = ROOT / "reports"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13, "axes.labelsize": 11})

def paired(split):
    frame = pd.read_parquet(DATA / "manifests" / f"{split}_manifest.parquet")
    return frame.loc[(frame.source_dataset == "Fakeddit") & (frame.download_status == "verified_reused_or_processed")].copy().reset_index(drop=True)

def canonical_source_labels(raw):
    """Fakeddit raw 1=true; return 0=SOURCE_TRUE, 1=SOURCE_NON_TRUE."""
    return 1 - np.asarray(raw, dtype=int)

def source_names(values):
    return np.where(np.asarray(values) == 0, "SOURCE_TRUE", "SOURCE_NON_TRUE")

def show_frame(frame, rows=10):
    display(frame.head(rows))


## Weighted late fusion

`P_non_true = alpha × P_text_non_true + (1-alpha) × P_image_non_true`. The saved alpha is 0.41 for text and 0.59 for image. Alpha was selected on validation, so the displayed validation result is selection-set performance, not independent confirmation.

In [2]:
weighted=pd.read_parquet(REPORTS/'weighted_late_fusion_v1/validation_predictions.parquet');alpha=0.41;text_non=1-weighted.text_probability_fake;image_non=1-weighted.image_probability_fake;combined=alpha*text_non+(1-alpha)*image_non;print('Maximum reconstruction error:',float(np.abs(combined-(1-weighted.combined_probability_fake)).max()));print('All stored raw-label decisions reproduced:',bool(np.array_equal((combined<.5).astype(int),weighted.prediction.to_numpy())))
metrics=json.loads((REPORTS/'weighted_late_fusion_v1/metrics.json').read_text());display(pd.DataFrame(metrics['validation']).T)

Maximum reconstruction error: 6.347894676572707e-08
All stored raw-label decisions reproduced: True


,accuracy,macro_f1,brier,log_loss,ece_10
image,0.693462,0.686088,0.203793,0.596260,0.061402
text,0.721329,0.718410,0.192448,0.571875,0.094445
alpha_0_5,0.756699,0.752744,0.176769,0.534793,0.110372
selected,0.764202,0.759459,0.178482,0.538431,0.105777


## CLIP gated feature fusion

The archived head combines frozen 512-D CLIP image and text features. A learned sigmoid gate is feature-wise, not scalar: `g = sigmoid(W[z_text;z_image]+b)` and `z_fused = g*z_text + (1-g)*z_image`. This differs from weighted late fusion, which combines final class probabilities; neither is cross-attention.

In [3]:
checkpoint=torch.load(REPORTS/'multimodal_fusion/clip_fusion_v1/gated/best_model.pt',map_location='cpu',weights_only=False);print('Gated configuration:',checkpoint['config']);print('Feature statistics shape:',tuple(checkpoint['feature_mean'].shape));gated=pd.read_parquet(REPORTS/'multimodal_fusion/clip_fusion_v1/gated/validation_predictions.parquet');print('Gated validation records:',len(gated))

Gated configuration: {'dataset_root': 'data/research_dataset', 'output_dir': 'reports/multimodal_fusion/clip_fusion_v1', 'config_copy_path': 'reports/multimodal_fusion/clip_fusion_v1/config.json', 'clip_model': 'openai/clip-vit-base-patch32', 'seed': 42, 'hidden_dim': 256, 'dropout': 0.2, 'batch_size': 64, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'epochs': 20, 'patience': 4}
Feature statistics shape: (1024,)
Gated validation records: 933


In [4]:
from sklearn.metrics import confusion_matrix,f1_score
y=canonical_source_labels(gated.label);p=canonical_source_labels(gated.prediction);cm=confusion_matrix(y,p,labels=[0,1]);print(cm.tolist(),'macro-F1',f1_score(y,p,average='macro'));display(pd.DataFrame(cm,index=['SOURCE_TRUE','SOURCE_NON_TRUE'],columns=['SOURCE_TRUE','SOURCE_NON_TRUE']))

[[327, 50], [67, 489]] macro-F1 0.8706998560844305


,SOURCE_TRUE,SOURCE_NON_TRUE
SOURCE_TRUE,327,50
SOURCE_NON_TRUE,67,489


## Application status

The gated model is preserved but archived/inactive in the application. It must not be silently substituted for the active text-only policy.